# Query Parquet Files - GraphRAG Entities

This notebook demonstrates how to query and analyze the `entities.parquet` file from GraphRAG using different methods including DuckDB, Pandas, and pattern matching.

## 1. Import Required Libraries

Import necessary libraries for working with Parquet files.

In [ ]:
# Import Required Libraries
import pandas as pd
import pyarrow.parquet as pq
import os
from IPython.display import display, Markdown

# Set display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

# Set the path to the parquet file
parquet_path = '/home/chuaxu/projects/graphrag/ragtest/output/entities.parquet'

# Verify the file exists
if os.path.exists(parquet_path):
    print("✅ Parquet file found")
    print(f"📁 Path: {parquet_path}")
else:
    print(f"❌ File not found: {parquet_path}")

## 2. Load Parquet File

Load the entities.parquet file into a DataFrame using pandas.

In [ ]:
# Load the parquet file into a DataFrame
df = pd.read_parquet(parquet_path)

print("=" * 80)
print("✅ SUCCESSFULLY LOADED PARQUET FILE")
print("=" * 80)
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 80)

## 3. Explore Data Structure

Examine the DataFrame structure, including column names, data types, and sample data.

In [ ]:
# Display column information
print("=" * 80)
print("📋 COLUMNS & DATA TYPES")
print("=" * 80)
for idx, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{idx:2d}. {col:20s} → {dtype}")
print("=" * 80)

In [ ]:
# Display first few rows
print("=" * 80)
print("🔍 FIRST 5 ROWS")
print("=" * 80)
display(df.head())

In [ ]:
# Display summary statistics for numeric columns
print("=" * 80)
print("📊 SUMMARY STATISTICS (Numeric Columns)")
print("=" * 80)
display(df.describe())

In [ ]:
# Display number of unique values in each column
print("=" * 80)
print("🔢 UNIQUE VALUES COUNT PER COLUMN")
print("=" * 80)
print(f"\n{'Column Name':<25} {'Unique Count':>15} {'Total Rows':>15}")
print("-" * 80)

for col in df.columns:
    try:
        unique_count = df[col].nunique()
        total_count = len(df)
        print(f"{col:<25} {unique_count:>15,} {total_count:>15,}")
    except (TypeError, ValueError) as e:
        # Skip columns with unhashable types (like arrays)
        print(f"{col:<25} {'N/A (array type)':>15} {len(df):>15,}")

print("-" * 80)
print(f"{'TOTAL COLUMNS':<25} {len(df.columns):>15,}")
print("=" * 80)

## 4. Filter and Query Data

Apply filters to extract specific entities like "Scrooge".

In [ ]:
# ⚙️ CONFIGURATION: Set the entity you want to search for
# Change this variable to search for different entities
SEARCH_ENTITY = 'THE GHOST'  # Change this to any entity name you want to find

print("=" * 80)
print(f"🔧 SEARCH CONFIGURATION")
print("=" * 80)
print(f"📝 Searching for: '{SEARCH_ENTITY}'")
print("=" * 80)

In [ ]:
# Method 1: Query for exact match
SEARCH_ENTITY = 'JACOB MARLEY'
print("=" * 80)
print(f"🔍 FILTER 1: Exact match for '{SEARCH_ENTITY}'")
print("=" * 80)
result = df[df['title'] == SEARCH_ENTITY]
print(f"✅ Found {len(result)} row(s)\n")

if len(result) > 0:
    display(result)
else:
    print(f"❌ No exact matches found for '{SEARCH_ENTITY}'")
    print("💡 Try the case-insensitive search below!")
print("=" * 80)

In [ ]:
# Method 1: Query for case-insensitive match (contains text)
SEARCH_ENTITY = 'GHOST'
print("=" * 80)
print(f"🔍 FILTER 1: Case-insensitive search for '{SEARCH_ENTITY}'")
print("=" * 80)

# Case-insensitive search - matches 'GHOST', 'Ghost', 'ghost', etc.
result = df[df['title'].str.contains(SEARCH_ENTITY, case=False, na=False)]
print(f"✅ Found {len(result)} row(s)\n")

if len(result) > 0:
    display(result)
    
    # Display full description content
    print("\n" + "=" * 80)
    print("📝 FULL DESCRIPTION CONTENT")
    print("=" * 80)
    for idx, desc in enumerate(result['description'].values, 1):
        print(f"\nDescription {idx}:")
        print("-" * 80)
        print(desc)
        print("-" * 80)
else:
    print(f"❌ No matches found for '{SEARCH_ENTITY}'")
    print("💡 Try adjusting your search term!")
print("=" * 80)

In [ ]:
# Method 2: Query for multiple entities
print("=" * 80)
print("🔍 FILTER 2: Multiple entities")
print("=" * 80)
entities_to_find = ['SCROOGE', 'JACOB MARLEY', 'GHOST', 'Scrooge', 'Jacob Marley', 'Ghost']
filtered = df[df['title'].isin(entities_to_find)]
print(f"✅ Found {len(filtered)} row(s)\n")

if len(filtered) > 0:
    display(filtered[['title', 'type', 'frequency', 'degree', 'description']])
else:
    print("❌ No matches found")
print("=" * 80)

In [ ]:
# Method 3: Query with pattern matching (case-insensitive)
print("=" * 80)
print("🔍 FILTER 3: Pattern matching (contains 'scrooge', case-insensitive)")
print("=" * 80)
pattern_match = df[df['title'].str.contains('scrooge', case=False, na=False)]
print(f"✅ Found {len(pattern_match)} row(s)\n")

if len(pattern_match) > 0:
    display(pattern_match[['title', 'type', 'frequency', 'degree', 'description']])
else:
    print("❌ No matches found")
print("=" * 80)

In [ ]:
# Method 4: Query using .loc[] with multiple conditions
print("=" * 80)
print("🔍 FILTER 4: High-frequency entities (frequency > 5)")
print("=" * 80)

if 'frequency' in df.columns:
    high_freq = df.loc[df['frequency'] > 5, ['title', 'type', 'frequency', 'degree']]
    print(f"✅ Found {len(high_freq)} entity/entities with frequency > 5\n")
    if len(high_freq) > 0:
        display(high_freq.head(10))
else:
    print("❌ 'frequency' column not found in dataframe")
    
print("=" * 80)

## 5. Aggregate and Analyze Data

Perform aggregations and analysis on the queried data.

In [ ]:
# Aggregate: Group by type and count
print("=" * 80)
print("📊 AGGREGATION 1: Count by Entity Type")
print("=" * 80)

if 'type' in df.columns:
    type_counts = df['type'].value_counts()
    print(f"\n{'Entity Type':<30} {'Count':>10}")
    print("-" * 42)
    for entity_type, count in type_counts.head(15).items():
        print(f"{str(entity_type):<30} {count:>10,}")
    print("-" * 42)
    print(f"{'TOTAL':<30} {type_counts.sum():>10,}")
else:
    print("❌ 'type' column not found")
    
print("=" * 80)

In [ ]:
# Aggregate: Statistical analysis
print("=" * 80)
print("📊 AGGREGATION 2: Statistical Summary")
print("=" * 80)

print("\n📈 FREQUENCY STATISTICS:")
print("-" * 40)
if 'frequency' in df.columns:
    print(f"  Mean:     {df['frequency'].mean():>10.2f}")
    print(f"  Median:   {df['frequency'].median():>10.2f}")
    print(f"  Min:      {df['frequency'].min():>10,}")
    print(f"  Max:      {df['frequency'].max():>10,}")
    print(f"  Std Dev:  {df['frequency'].std():>10.2f}")
else:
    print("  ❌ 'frequency' column not found")

print("\n📈 DEGREE STATISTICS:")
print("-" * 40)
if 'degree' in df.columns:
    print(f"  Mean:     {df['degree'].mean():>10.2f}")
    print(f"  Median:   {df['degree'].median():>10.2f}")
    print(f"  Min:      {df['degree'].min():>10,}")
    print(f"  Max:      {df['degree'].max():>10,}")
    print(f"  Std Dev:  {df['degree'].std():>10.2f}")
else:
    print("  ❌ 'degree' column not found")
    
print("=" * 80)

## 6. Query Using DuckDB (Fast SQL Queries)

Use DuckDB for efficient SQL-based queries on the parquet file.

In [ ]:
# DuckDB SQL Query 1: Find Scrooge (case-insensitive)
try:
    import duckdb
    
    print("=" * 80)
    print("🦆 DUCKDB SQL QUERY 1: Find 'Scrooge' (case-insensitive)")
    print("=" * 80)
    
    result = duckdb.query(f"""
        SELECT title, type, frequency, degree, description
        FROM '{parquet_path}'
        WHERE LOWER(title) LIKE '%scrooge%'
        LIMIT 1000
    """).to_df()
    
    print(f"✅ Found {len(result)} row(s)\n")
    if len(result) > 0:
        display(result)
    else:
        print("❌ No matches found")
    print("=" * 80)
    
except ImportError:
    print("=" * 80)
    print("❌ DuckDB not installed")
    print("💡 Install with: pip install duckdb")
    print("=" * 80)

In [ ]:
# DuckDB SQL Query 2: Multiple entities with aggregation
try:
    import duckdb
    
    print("=" * 80)
    print("🦆 DUCKDB SQL QUERY 2: Top 10 entities by frequency")
    print("=" * 80)
    
    result = duckdb.query(f"""
        SELECT 
            title,
            type,
            frequency,
            degree,
            SUBSTR(description, 1, 100) as description_preview
        FROM '{parquet_path}'
        ORDER BY frequency DESC
        LIMIT 10
    """).to_df()
    
    print(f"✅ Found {len(result)} row(s)\n")
    display(result)
    print("=" * 80)
    
except ImportError:
    pass

## 7. Export Query Results

Export the query results to various formats (CSV, JSON, Parquet).

In [ ]:
# Export filtered results to CSV
scrooge_data = df[df['title'].str.contains('scrooge', case=False, na=False)]

print("=" * 80)
print("💾 EXPORT RESULTS")
print("=" * 80)

if len(scrooge_data) > 0:
    # Export to CSV
    csv_path = '/home/chuaxu/projects/graphrag/scrooge_entities.csv'
    scrooge_data.to_csv(csv_path, index=False)
    print(f"✅ Exported to CSV: {csv_path}")
    
    # Export to JSON
    json_path = '/home/chuaxu/projects/graphrag/scrooge_entities.json'
    scrooge_data.to_json(json_path, orient='records', indent=2)
    print(f"✅ Exported to JSON: {json_path}")
    
    # Export to Parquet
    parquet_output = '/home/chuaxu/projects/graphrag/scrooge_entities.parquet'
    scrooge_data.to_parquet(parquet_output, index=False)
    print(f"✅ Exported to Parquet: {parquet_output}")
    
    print(f"\n📊 Exported {len(scrooge_data)} row(s)")
else:
    print("❌ No data to export")
    
print("=" * 80)